# Занятие 4, демо 2. Шаг солвера и вызов сети - разные вещи

«Сделали в четыре раза меньше шагов» и «вызвали сеть в четыре раза реже» - не
одно и то же утверждение. Считать надо вызовы: платим мы за них.

Поле то же, что в демо 1: прямая интерполяция занятия 4, генерация из
$\tau=0$ в $\tau=1$. Веса загружаем готовыми, чтобы не ждать обучение второй раз.

In [ ]:
"""Общая база демо занятий 4 и 5: данные, сеть, метрика, солверы.

Здесь намеренно нет ни одной конструкции пути. На занятии 4 время идёт от шума
к данным, на занятии 5 - наоборот, и если держать обе ориентации в одном файле,
одно и то же имя начинает означать разное. Поэтому пути лежат по отдельности:
flow_cfm.py - занятие 4, flow_vp.py - занятие 5.

Направление интегрирования солверы получают аргументами t_from и t_to, а не
берут из умолчания: так его видно в месте вызова.

Датасет, сеть и расписание обучения те же, что в ДЗ-2, - иначе сравнение
занятий между собой перестало бы быть корректным.
"""
import math

import torch
from torch import nn

# --------------------------------------------------------------------------
# Данные: восемь гауссиан на окружности
# --------------------------------------------------------------------------

N_MODES = 8
RING_RADIUS = 2.0
MODE_STD = 0.15


def mode_centers(dtype=torch.float32) -> torch.Tensor:
    """Центры восьми компонент, форма [8, 2]."""
    k = torch.arange(N_MODES, dtype=dtype)
    angle = math.pi * k / 4
    return RING_RADIUS * torch.stack([torch.cos(angle), torch.sin(angle)], dim=-1)


def sample_data(n: int, generator: torch.Generator,
                dtype=torch.float32) -> torch.Tensor:
    """n точек из смеси восьми гауссиан, форма [n, 2]."""
    centers = mode_centers(dtype=dtype)
    which = torch.randint(N_MODES, (n,), generator=generator)
    noise = torch.randn(n, 2, generator=generator, dtype=dtype)
    return centers[which] + MODE_STD * noise


# --------------------------------------------------------------------------
# Сеть
# --------------------------------------------------------------------------

class VelocityNet(nn.Module):
    """Маленькая сеть: вход (x, tau) -> вектор в R^2. 4546 параметров."""

    def __init__(self, width: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, width),
            nn.SiLU(),
            nn.Linear(width, width),
            nn.SiLU(),
            nn.Linear(width, 2),
        )

    def forward(self, x: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([x, tau], dim=-1))


def make_model(seed: int = 0, width: int = 64) -> VelocityNet:
    """Сеть с воспроизводимой инициализацией."""
    state = torch.get_rng_state()
    try:
        torch.manual_seed(seed)
        model = VelocityNet(width=width)
    finally:
        torch.set_rng_state(state)
    return model


# --------------------------------------------------------------------------
# Метрика: energy distance
# --------------------------------------------------------------------------

def energy_distance(x: torch.Tensor, y: torch.Tensor) -> float:
    """Оценка energy distance между двумя выборками. Считается в float64."""
    x = x.detach().to(torch.float64)
    y = y.detach().to(torch.float64)
    n, m = x.shape[0], y.shape[0]
    cross = torch.cdist(x, y).sum() / (n * m)
    inner_x = torch.cdist(x, x).sum() / (n * n)
    inner_y = torch.cdist(y, y).sum() / (m * m)
    return float(2 * cross - inner_x - inner_y)


# --------------------------------------------------------------------------
# Солверы и счётчик вызовов
# --------------------------------------------------------------------------

class CountingField:
    """Обёртка над полем, считающая вызовы."""

    def __init__(self, field):
        self.field = field
        self.calls = 0

    def __call__(self, x: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        return self.field(x, tau)


def euler_path(field, x: torch.Tensor, t_from: float, t_to: float,
               steps: int) -> torch.Tensor:
    """Явный Эйлер из t_from в t_to. Один шаг - один вызов поля.

    Возвращает все точки, форма [steps + 1, n, d].
    """
    if not isinstance(steps, int) or isinstance(steps, bool) or steps <= 0:
        raise ValueError("steps должно быть положительным int")

    h = (t_to - t_from) / steps
    ones = torch.ones(x.shape[0], 1, dtype=x.dtype, device=x.device)
    points = [x.clone()]

    with torch.no_grad():
        for m in range(steps):
            x = x + h * field(x, ones * (t_from + m * h))
            points.append(x.clone())

    return torch.stack(points)


def rk4_path(field, x: torch.Tensor, t_from: float, t_to: float,
             steps: int) -> torch.Tensor:
    """Классический RK4 из t_from в t_to. Один шаг - четыре вызова поля.

    Возвращает точки на границах макрошагов, форма [steps + 1, n, d].
    """
    if not isinstance(steps, int) or isinstance(steps, bool) or steps <= 0:
        raise ValueError("steps должно быть положительным int")

    h = (t_to - t_from) / steps
    ones = torch.ones(x.shape[0], 1, dtype=x.dtype, device=x.device)
    points = [x.clone()]

    with torch.no_grad():
        for m in range(steps):
            t = t_from + m * h
            k1 = field(x, ones * t)
            k2 = field(x + 0.5 * h * k1, ones * (t + 0.5 * h))
            k3 = field(x + 0.5 * h * k2, ones * (t + 0.5 * h))
            k4 = field(x + h * k3, ones * (t + h))
            x = x + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
            points.append(x.clone())

    return torch.stack(points)


def euler_solve(field, x: torch.Tensor, t_from: float, t_to: float,
                nfe: int) -> torch.Tensor:
    """Эйлер с бюджетом nfe вызовов поля: это ровно nfe шагов."""
    return euler_path(field, x, t_from, t_to, nfe)[-1]


def rk4_solve(field, x: torch.Tensor, t_from: float, t_to: float,
              nfe: int) -> torch.Tensor:
    """RK4 с бюджетом nfe вызовов поля: это nfe // 4 макрошагов."""
    if not isinstance(nfe, int) or isinstance(nfe, bool) or nfe <= 0:
        raise ValueError("nfe должно быть положительным int")
    if nfe % 4:
        raise ValueError("для RK4 бюджет вызовов должен делиться на 4")
    return rk4_path(field, x, t_from, t_to, nfe // 4)[-1]


# --------------------------------------------------------------------------
# Обучающий драйвер
# --------------------------------------------------------------------------

TRAIN_CONFIG = {
    "batch_size": 512,
    "steps": 12000,
    "lr": 2e-3,
    "data_seed": 1234,
    "noise_seed": 5678,
    "model_seed": 0,
    "report_every": 2000,
}


# --------------------------------------------------------------------------
# Замороженные веса: кладём вместе с тем, чем они являются
# --------------------------------------------------------------------------

def save_checkpoint(path, model, objective: str, cfg, history) -> None:
    """Сохраняет веса вместе с конструкцией, конфигом и достигнутыми потерями."""
    torch.save({"state_dict": model.state_dict(),
                "objective": objective,
                "config": dict(cfg),
                "final_loss": sum(history[-500:]) / 500}, path)


def load_checkpoint(path, objective: str):
    """Загружает веса и проверяет, что это обещанная конструкция.

    Демо занятий 4 и 5 стоят на утверждении «одна архитектура, один бюджет
    обучения». Утверждение, которое нельзя проверить на месте, - это дыра:
    файл легко перепутать, и ошибка будет молчаливой. Поэтому конструкция
    лежит внутри файла и сверяется при загрузке.
    """
    blob = torch.load(path, map_location="cpu", weights_only=False)
    if blob["objective"] != objective:
        raise ValueError(f"{path}: обучено на '{blob['objective']}', "
                         f"ожидалось '{objective}'")
    model = make_model(seed=blob["config"]["model_seed"])
    model.load_state_dict(blob["state_dict"])
    model.eval()
    return model, blob


"""Занятие 4: прямая интерполяция и conditional flow matching.

Ориентация занятия 4: `noise` - источник, `data` - цель, tau идёт от 0 к 1,
генерация вперёд. Имя x_0 здесь не используется намеренно: на занятии 5 оно
означает ровно противоположное.

Требует flow_common.py.
"""
import torch



def straight_interpolation(noise, data, tau):
    """Прямой отрезок между источником и целью."""
    return (1.0 - tau) * noise + tau * data


def cfm_target(noise, data):
    """Производная вдоль отрезка: она постоянна и равна разности концов."""
    return data - noise


def cfm_loss(model, noise, data, tau):
    """Conditional flow matching на прямой интерполяции."""
    prediction = model(straight_interpolation(noise, data, tau), tau)
    return ((prediction - cfm_target(noise, data)) ** 2).sum(dim=-1).mean()


def train_cfm(cfg=None, verbose=True):
    """Обучение поля на прямой интерполяции. Возвращает (model, история)."""
    cfg = dict(TRAIN_CONFIG if cfg is None else cfg)
    model = make_model(seed=cfg["model_seed"])
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    data_gen = torch.Generator().manual_seed(cfg["data_seed"])
    noise_gen = torch.Generator().manual_seed(cfg["noise_seed"])
    batch = cfg["batch_size"]
    history = []

    for step in range(1, cfg["steps"] + 1):
        data = sample_data(batch, data_gen)
        noise = torch.randn(batch, 2, generator=noise_gen)
        tau = torch.rand(batch, 1, generator=noise_gen)

        loss = cfm_loss(model, noise, data, tau)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        history.append(loss.detach().item())

        if verbose and step % cfg["report_every"] == 0:
            window = history[-cfg["report_every"]:]
            print(f"  шаг {step:6d}   потери {sum(window) / len(window):.4f}")

    return model, history


def learned_field(model):
    """Поле ОДУ занятия 4: предсказание сети как есть, без множителей."""
    def field(x, tau):
        return model(x, tau)
    return field


def oracle_field(x, tau):
    """Точное population-optimal поле E[data - noise | x_tau = x].

    Нейросети здесь нет: для нашей смеси гауссиан условное среднее выписывается
    аналитически. Пусть a = 1 - tau, b = tau, s = MODE_STD, D = a^2 + b^2 s^2.
    Тогда веса компонент пропорциональны exp(-||x - b mu_k||^2 / 2D), а внутри
    компоненты условное среднее равно mu_k + (b s^2 - a) / D * (x - b mu_k).

    Нужно, чтобы отделить свойство конструкции от ошибки обучения: если кривые
    траектории видны и здесь, дело не в том, что сеть маленькая или недоучена.
    """
    a = 1.0 - tau                                          # [n, 1]
    b = tau                                                # [n, 1]
    mu = mode_centers(dtype=x.dtype)                       # [8, 2]
    var = a * a + b * b * MODE_STD ** 2                    # [n, 1]
    shift = x.unsqueeze(1) - b.unsqueeze(1) * mu           # [n, 8, 2]
    weights = torch.softmax(-(shift ** 2).sum(-1) / (2 * var), dim=-1)
    coefficient = ((b * MODE_STD ** 2 - a) / var).unsqueeze(1)
    per_mode = mu.unsqueeze(0) + coefficient * shift       # [n, 8, 2]
    return (weights.unsqueeze(-1) * per_mode).sum(dim=1)


def path_ratio(points):
    """Длина пути, делённая на длину хорды. Единица - прямая."""
    length = (points[1:] - points[:-1]).norm(dim=-1).sum(dim=0)
    chord = (points[-1] - points[0]).norm(dim=-1)
    return length / chord


def cfm_sample(model, noise, nfe):
    """Генерация полем занятия 4: Эйлер из tau=0 в tau=1."""
    return euler_solve(learned_field(model), noise, 0.0, 1.0, nfe)

In [ ]:
model, blob = load_checkpoint("cfm_seed0.pt", "cfm")
field = learned_field(model)
print(f"поле занятия 4: {blob['objective']}, "
      f"шагов обучения {blob['config']['steps']}")

## Одинаковое число шагов - разная цена

Обёртка `CountingField` считает каждый вызов поля. Даём обоим солверам
одинаковое число **макрошагов** и смотрим на счётчик.

In [ ]:
z = torch.randn(512, 2, generator=torch.Generator().manual_seed(5))

print(f"{'солвер':<8} {'макрошагов':>11} {'вызовов сети':>14}")
for name, run in (("euler", euler_path), ("rk4", rk4_path)):
    counter = CountingField(field)
    run(counter, z, 0.0, 1.0, 8)
    print(f"{name:<8} {8:>11} {counter.calls:>14}")

## Теперь при равном бюджете вызовов

Сравниваем при одинаковом $N_{\mathrm{FE}}$: у RK4 это вчетверо меньше
макрошагов. Две метрики рядом:

- **ED** - energy distance между выборкой солвера и выборкой данных. Это
  качество результата целиком: и ошибка солвера, и ошибка обученного поля;
- **откл.** - среднее расстояние до почти точного решения **того же** ОДУ
  ($N_{\mathrm{FE}}=4096$, RK4). Это уже только ошибка солвера.

Вторая колонка нужна затем, чтобы не приписать полю то, что делает солвер, и
наоборот.

In [ ]:
data_sample = sample_data(2048, torch.Generator().manual_seed(99))
z_big = torch.randn(2048, 2, generator=torch.Generator().manual_seed(11))
exact = rk4_solve(field, z_big, 0.0, 1.0, 4096)

print(f"ED(почти точное решение ОДУ, данные) = "
      f"{energy_distance(exact, data_sample):.4f}")
print()
print(f"{'вызовов':>8}  {'euler: ED':>10} {'откл.':>8}"
      f"  {'rk4: ED':>10} {'откл.':>8}")
for nfe in (4, 8, 16, 32):
    y_e = euler_solve(field, z_big, 0.0, 1.0, nfe)
    y_r = rk4_solve(field, z_big, 0.0, 1.0, nfe)
    print(f"{nfe:>8}  {energy_distance(y_e, data_sample):>10.4f}"
          f" {float((y_e - exact).norm(dim=-1).mean()):>8.4f}"
          f"  {energy_distance(y_r, data_sample):>10.4f}"
          f" {float((y_r - exact).norm(dim=-1).mean()):>8.4f}")

## Что из этого следует

При равном числе **шагов** RK4 тратит вчетверо больше вызовов сети. Такое
сравнение отвечает на вопрос о точности схемы на шаг, но не на вопрос о качестве
при равных затратах - а платим мы именно за вызовы.

При равном числе **вызовов** видно две разные вещи. На четырёх вызовах у RK4
остаётся один-единственный макрошаг, и он проигрывает. Начиная с восьми он
выигрывает, а с шестнадцати его ED упирается в строку над таблицей - в качество
самого обученного поля. Что это упор именно в поле, а не в солвер, видно по
колонке «откл.»: у RK4 она падает до 0.0002, то есть решение ОДУ уже практически
точное, а ED при этом не двигается с 0.0050.

Картина устойчива: на трёх других наборах шума порядок ровно тот же, меняется
только уровень плато.

Отсюда узкое и точное правило: **сравнивая солверы для одного и того же поля,
фиксируем $N_{\mathrm{FE}}$**, а не число шагов. Число макрошагов - внутренняя
деталь схемы. Между разными сетями и один вызов стоит по-разному, поэтому на
такое сравнение это правило не переносится: там честнее FLOPs или время.